# Retrieval
Implements the **Phase 2 retrieval step** of Vectorless RAG:

1. Reload the tree (`Tree.treeJson`) built by `treeBuilder.ipynb`.
2. Use the LLM to **traverse** the tree (Root → Chapter → Section) guided by the user query.
3. Pull the **candidate pages** that live inside the winning section.
4. Ask the LLM to **re-rank** those candidates by relevance to the query.
5. Fetch the **full page content** for the top-ranked pages from Postgres.

No embeddings/vector DB are used anywhere — every retrieval decision is made by the LLM
reasoning over titles/summaries/keywords stored in `Page.metadata` and `Tree.treeJson`.

**Run order:** `pageMetadata.ipynb` → `treeBuilder.ipynb` → `retreivalPDF.ipynb`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. Imports & Connection

In [2]:
import json
import re

from src.config.db import get_connection
from src.config.llm import llm

conn = get_connection()

## 2. Set Document & Query
Change `document_id` to the document you built a tree for.
`query` is the user's natural-language question.

In [3]:
document_id = "DOC000001"
query = "What are financial vulnerabilities?"

## 3. Tree Node Helpers
Same lightweight `TreeNode` wrapper + `load_tree_from_db` used in `treeBuilder.ipynb`.
Kept here so this notebook can run standalone.

In [4]:
class TreeNode:
    """Lightweight wrapper around a tree dict node."""

    def __init__(self, data: dict, parent=None):
        self.data = data
        self.parent = parent
        self.children = []

    @property
    def level(self):
        return self.data.get("level", 0)

    @property
    def title(self):
        return self.data.get("title", "")

    @property
    def summary(self):
        return self.data.get("summary", "")

    @property
    def path(self):
        return self.data.get("path", "")

    @property
    def type(self):
        return self.data.get("type", "root")

    def __repr__(self):
        return f"TreeNode(level={self.level}, path={self.path!r}, title={self.title!r})"


def build_tree_nodes(tree_dict: dict, parent=None) -> "TreeNode":
    """Recursively build TreeNode objects from the JSON tree dict."""
    node = TreeNode(tree_dict, parent)
    for child_dict in tree_dict.get("children", []):
        child_node = build_tree_nodes(child_dict, parent=node)
        node.children.append(child_node)
    return node


def load_tree_from_db(conn, document_id: str) -> "TreeNode":
    """Load treeJson from DB and return the root TreeNode."""
    with conn.cursor() as cur:
        cur.execute(
            'SELECT "treeJson" FROM "Tree" WHERE "documentId" = %s',
            (document_id,),
        )
        row = cur.fetchone()

    if row is None:
        raise ValueError(f"No tree found for documentId={document_id!r}")

    tree_dict = row[0]
    if isinstance(tree_dict, str):
        tree_dict = json.loads(tree_dict)

    return build_tree_nodes(tree_dict)

## 4. Load the Tree

In [5]:
tree_root = load_tree_from_db(conn, document_id)

print("Root:", tree_root.title)
print(f"Chapters: {len(tree_root.children)}")
for ch in tree_root.children:
    print(f"  [{ch.path}] {ch.title}  ({len(ch.children)} sections)")

Root: 2023-annual-report-truncated
Chapters: 5
  [1] Introduction  (5 sections)
  [2] Financial Stability  (4 sections)
  [3] Supervision and Regulation  (3 sections)
  [4] Examinations and Inspections  (2 sections)
  [5] Financial Stability and Regulatory Reports  (1 sections)


## 5. LLM-Guided Tree Traversal
At every level the LLM is shown the current node's children (title + summary)
and asked which one best matches the query. We keep descending until we hit
a leaf (a `section` node, which has no children).

The traversal is defensive: if the LLM's reply can't be parsed into a valid
child index, we fall back to child `1` instead of crashing.

In [6]:
def _parse_choice(response_text: str, num_children: int) -> int:
    """Extract the first integer in range [1, num_children] from the LLM reply."""
    match = re.search(r"\d+", response_text)
    if match:
        choice = int(match.group())
        if 1 <= choice <= num_children:
            return choice
    return 1  # safe fallback


def choose_child(query: str, current: "TreeNode", llm) -> int:
    """Ask the LLM which child of `current` is most relevant to `query`. Returns 1-based index."""
    children = current.children

    prompt = f"""You are navigating a document tree to answer a user's question.

User Query:
{query}

Current Node ({current.type}):
{current.title}

Available Children:
"""

    for i, child in enumerate(children):
        prompt += f"""
{i + 1}. [{child.type}] {child.title}
   Summary: {child.summary}
"""

    prompt += """
Which child is most likely to contain the answer?
Return ONLY the number of the child. No explanation, no punctuation.
"""

    response = llm.invoke(prompt).content.strip()
    return _parse_choice(response, len(children))


def traverse_tree(query: str, root: "TreeNode", llm) -> dict:
    """
    Walk root -> chapter -> section, letting the LLM pick a child at each level.
    Returns the path taken (list of TreeNode) and the leaf node reached.
    """
    current = root
    path = [root]

    while current.children:
        choice = choose_child(query, current, llm)
        current = current.children[choice - 1]
        path.append(current)

    return {"path": path, "leaf": current}

## 6. Run Traversal

In [7]:
traversal = traverse_tree(query, tree_root, llm)
leaf = traversal["leaf"]

print("Traversal path:")
for node in traversal["path"]:
    print(f"  [{node.type}] {node.title}  (path={node.path})")

print()
print(f"Leaf section pages: {leaf.data.get('pageStart')}-{leaf.data.get('pageEnd')}")
print(f"pageIds in leaf: {len(leaf.data.get('pageIds', []))}")

Traversal path:
  [root] 2023-annual-report-truncated  (path=root)
  [chapter] Financial Stability  (path=2)
  [section] Monitoring Financial Vulnerabilities  (path=2.2)

Leaf section pages: 23-24
pageIds in leaf: 2


## 7. Candidate Pages from the Leaf Section
Fetch the metadata (title/summary/keywords) for every page inside the winning
section, so the LLM can re-rank them without needing full page text yet.

In [8]:
def get_candidate_pages(conn, leaf: "TreeNode") -> list:
    """Fetch pageNumber + metadata for every page inside the leaf section."""
    page_ids = leaf.data.get("pageIds", [])
    if not page_ids:
        return []

    placeholders = ",".join(["%s"] * len(page_ids))
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT "pageNumber", metadata
            FROM "Page"
            WHERE id IN ({placeholders})
            ORDER BY "pageNumber"
            ''',
            page_ids,
        )
        rows = cur.fetchall()

    candidates = []
    for page_number, metadata in rows:
        meta = metadata or {}
        candidates.append({
            "pageNumber": page_number,
            "title": meta.get("title", ""),
            "summary": meta.get("summary", ""),
            "keywords": meta.get("keywords", []),
        })
    return candidates


candidates = get_candidate_pages(conn, leaf)
print(f"Fetched {len(candidates)} candidate pages")
candidates[:3]

Fetched 2 candidate pages


[{'pageNumber': 23,
  'title': 'Financial Stability Monitoring Framework',
  'summary': "The Federal Reserve's monitoring framework distinguishes between shocks to and vulnerabilities of the financial system, and maintains a flexible, forward-looking financial stability monitoring program.",
  'keywords': ['financial stability',
   'monitoring framework',
   'Federal Reserve']},
 {'pageNumber': 24,
  'title': 'Asset Valuation Pressures',
  'summary': 'Overvalued assets are a vulnerability because the unwinding of high prices can be destabilizing, and the Federal Reserve tracks a broad range of measures to assess asset valuation pressures.',
  'keywords': ['asset valuation pressures',
   'Federal Reserve',
   'financial stability']}]

## 8. LLM Re-ranking of Candidates
Ask the LLM to order the candidate pages from most to least relevant to the
query. The response is parsed defensively — any non-numeric lines are
ignored, duplicates are dropped, and page numbers outside the candidate set
are discarded.

In [9]:
def rank_pages(query: str, candidates: list, llm) -> list:
    """Return candidate page numbers ordered from most to least relevant."""
    if not candidates:
        return []

    valid_pages = {c["pageNumber"] for c in candidates}

    prompt = f"""User Question:
{query}

Candidate Pages:
"""

    for page in candidates:
        prompt += f"""
Page {page['pageNumber']}
Title: {page['title']}
Summary: {page['summary']}
Keywords: {", ".join(page['keywords'])}
"""

    prompt += """
Return ONLY the page numbers, ranked from most relevant to least relevant,
one per line. No explanation, no extra text.

Example:
112
114
113
"""

    response = llm.invoke(prompt).content

    ranked, seen = [], set()
    for line in response.splitlines():
        digits = re.findall(r"\d+", line.strip())
        for d in digits:
            page_num = int(d)
            if page_num in valid_pages and page_num not in seen:
                ranked.append(page_num)
                seen.add(page_num)

    # Fall back to original (unranked) order for any candidates the LLM missed
    for c in candidates:
        if c["pageNumber"] not in seen:
            ranked.append(c["pageNumber"])
            seen.add(c["pageNumber"])

    return ranked


ranked_pages = rank_pages(query, candidates, llm)
print("Ranked page numbers:", ranked_pages)

Ranked page numbers: [24, 23]


## 9. Fetch Full Content for Top-K Pages

In [10]:
def fetch_pages_content(conn, document_id: str, page_numbers: list) -> dict:
    """Fetch full page text for the given page numbers, keyed by pageNumber."""
    if not page_numbers:
        return {}

    placeholders = ",".join(["%s"] * len(page_numbers))
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT "pageNumber", content
            FROM "Page"
            WHERE "documentId" = %s
              AND "pageNumber" IN ({placeholders})
            ''',
            (document_id, *page_numbers),
        )
        rows = cur.fetchall()

    content_by_page = {page_number: content for page_number, content in rows}
    # Preserve the caller-supplied ranking order
    return {p: content_by_page[p] for p in page_numbers if p in content_by_page}


TOP_K = 3
top_pages = ranked_pages[:TOP_K]
pages_content = fetch_pages_content(conn, document_id, top_pages)

for page_number, content in pages_content.items():
    print(f"--- Page {page_number} ({len(content)} chars) ---")
    print(content[:200])
    print()

--- Page 24 (2581 chars) ---
Equity prices w ere still high relative to ear n-
ings ( figure 3.2 ). In addition, real estate prices
continued to be high relative to fundamentals.
Spreads on cor porate bonds and loans ended
2023 a

--- Page 23 (3228 chars) ---
Consistent with this vie w of financial stability , the Federal Reser ve Board’ s monitoring frame work
distinguishes betw een shocks to and vulnerabilities of the financial system. Shocks, such as
su



## 10. Full Retrieval Pipeline
Wraps steps 5-9 into a single reusable function. This is what
`answerGeneration.ipynb` and `evaluation.ipynb` call.

In [11]:
def retrieve(query: str, document_id: str, conn, llm, top_k: int = 3) -> dict:
    """
    End-to-end retrieval: tree traversal -> candidate pages -> re-rank -> fetch content.

    Returns a dict with:
      tree_path      - list of {type, title, path} dicts describing the traversal
      leaf           - the winning section's raw tree data
      candidates     - candidate page metadata from the leaf section
      ranked_pages   - all candidate page numbers, ranked
      top_pages      - the top_k page numbers actually used
      pages_content  - {pageNumber: full_text} for top_pages
    """
    tree_root = load_tree_from_db(conn, document_id)
    traversal = traverse_tree(query, tree_root, llm)
    leaf = traversal["leaf"]

    candidates = get_candidate_pages(conn, leaf)
    ranked_pages = rank_pages(query, candidates, llm)
    top_pages = ranked_pages[:top_k]
    pages_content = fetch_pages_content(conn, document_id, top_pages)

    return {
        "tree_path": [
            {"type": n.type, "title": n.title, "path": n.path}
            for n in traversal["path"]
        ],
        "leaf": leaf.data,
        "candidates": candidates,
        "ranked_pages": ranked_pages,
        "top_pages": top_pages,
        "pages_content": pages_content,
    }

## 11. Demo

In [12]:
result = retrieve(query, document_id, conn, llm, top_k=3)

print("Query:", query)
print()
print("Tree path:")
for node in result["tree_path"]:
    print(f"  [{node['type']}] {node['title']}")
print()
print("Top pages used for answer generation:", result["top_pages"])

Query: What are financial vulnerabilities?

Tree path:
  [root] 2023-annual-report-truncated
  [chapter] Financial Stability
  [section] Monitoring Financial Vulnerabilities

Top pages used for answer generation: [24, 23]
